<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Ex-Dividend_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import time
from IPython.display import display


## Ex-Dividend Date Price Drop Analysis

**Purpose:** Analyzes stock price behavior (specifically drops in Open and Low prices) relative to the dividend amount on ex-dividend dates for a list of S&P 500 stocks. It also includes market-adjusted drop calculations using SPY as a benchmark.

**Input Data:**
- `sp500_df` (DataFrame): A DataFrame containing S&P 500 stock symbols.
- Historical stock data from Yahoo Finance for individual tickers and SPY benchmark.

**Output Data:**
- `df_ex_dates` (DataFrame): A MultiIndex DataFrame containing detailed ex-dividend event records, including:
    - `Info`: Ticker, Ex_Dividend_Date, Dividend_Amount, Prev_Close, Div_Yield_Pct
    - `Open Metrics`: Ex_Open, Open_Price_Drop, Open_Drop_Pct_of_Div
    - `Low Metrics`: Ex_Low, Low_Price_Drop, Low_Drop_Pct_of_Div
    - `Market Adjusted`: SPY_Adjusted_Drop, Adj_Open_Drop_Pct_of_Div
- `stats_df` (DataFrame): A DataFrame summarizing the mean, median, standard deviation, and count of 'Open Drop (% of Div)' and 'Low Drop (% of Div)'.

**How it works:**
1. **Load S&P 500 Tickers:** Reads a CSV containing S&P 500 company symbols and processes them.
2. **Download Benchmark Data:** Fetches 2 years of historical data for the SPY ETF (benchmark) to calculate market adjustments.
3. **Iterate and Analyze:** For each S&P 500 ticker:
    - Downloads 2 years of historical stock data.
    - Identifies ex-dividend dates where a dividend was paid.
    - For each ex-dividend event:
        - Calculates the dividend yield percentage based on the previous day's closing price.
        - Computes the price drop from the previous close to the ex-dividend open (`Open_Price_Drop`).
        - Computes the price drop from the previous close to the ex-dividend low (`Low_Price_Drop`).
        - Expresses these drops as a percentage of the dividend amount (`Open_Drop_Pct_of_Div`, `Low_Drop_Pct_of_Div`).
        - Calculates a market-adjusted open drop by accounting for the SPY's overnight return on the same day.
        - Stores all calculated metrics in `ex_div_records`.
4. **Aggregate Results:** Concatenates all `ex_div_records` into a final DataFrame `df_ex_dates`.
5. **Summarize Statistics:** Calculates descriptive statistics (mean, median, std dev, count) for the `Open_Drop_Pct_of_Div` and `Low_Drop_Pct_of_Div` columns and displays them in `stats_df`.

In [27]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
SP500_id = '1gWxaB7UZfvHHGlQJoQRucvvO5eGBOCKY'
SP500 = f'https://drive.google.com/uc?export=download&id={SP500_id}'


#1KcMdKzzwwkcc7fLpPAe_cwqkw5TGBv8g
#1gWxaB7UZfvHHGlQJoQRucvvO5eGBOCKY



In [44]:
import pandas as pd
import yfinance as yf

# Read local CSV file
sp500_df = pd.read_csv(SP500)
tickers = sp500_df['Symbol'].astype(str).str.replace('.', '-', regex=False).tolist()

# 1. Download benchmark (SPY) data first for market-adjustment calculation
spy_hist = yf.Ticker("SPY").history(period="2y", auto_adjust=False)
spy_hist.index = spy_hist.index.tz_localize(None) # Normalize timezone

ex_div_records = []

# Process tickers (increase batch size safely)
for ticker in tickers[:503]:
    try:
        stock = yf.Ticker(ticker)
        # FORCE auto_adjust=False to get raw unadjusted transaction prices
        hist = stock.history(period="2y", auto_adjust=False)

        if hist.empty or 'Dividends' not in hist.columns:
            continue

        # Normalize timezones to prevent index lookup failures
        hist.index = hist.index.tz_localize(None)
        div_rows = hist[hist['Dividends'] > 0]

        for date, row in div_rows.iterrows():
            loc = hist.index.get_loc(date)

            # If get_loc returns a slice or mask due to duplicates, extract integer
            if isinstance(loc, slice):
                loc = loc.start
            elif not isinstance(loc, int):
                loc = int(loc[0])

            # Guard against first row edge-case (loc - 1 == -1)
            if loc <= 0:
                continue

            prev_row = hist.iloc[loc - 1]
            prev_close = prev_row['Close']
            ex_open = row['Open']
            ex_low = row['Low']
            div_amount = row['Dividends']

            if prev_close <= 0 or div_amount <= 0:
                continue

            # Yield calculation relative to previous close
            div_yield_pct = (div_amount / prev_close) * 100

            # --- OPEN DROP METRICS ---
            open_drop = prev_close - ex_open
            open_drop_pct_div = (open_drop / div_amount) * 100

            # --- LOW DROP METRICS ---
            low_drop = prev_close - ex_low
            low_drop_pct_div = (low_drop / div_amount) * 100

            # --- MARKET-ADJUSTED OPEN DROP (SPY Benchmark) ---
            # Measure SPY return on the exact same dates
            if date in spy_hist.index:
                spy_loc = spy_hist.index.get_loc(date)
                if isinstance(spy_loc, slice): spy_loc = spy_loc.start
                if spy_loc > 0:
                    spy_prev_close = spy_hist.iloc[spy_loc - 1]['Close']
                    spy_ex_open = spy_hist.loc[date, 'Open']
                    spy_return = (spy_ex_open - spy_prev_close) / spy_prev_close

                    # Deduct broad market overnight movement from raw drop
                    adjusted_open_drop = open_drop - (prev_close * spy_return)
                    adj_open_drop_pct_div = (adjusted_open_drop / div_amount) * 100
                else:
                    adjusted_open_drop, adj_open_drop_pct_div = None, None
            else:
                adjusted_open_drop, adj_open_drop_pct_div = None, None

            ex_div_records.append({
                ('Info', 'Ticker'): ticker,
                ('Info', 'Ex_Dividend_Date'): date.strftime('%Y-%m-%d'),
                ('Info', 'Dividend_Amount'): div_amount,
                ('Info', 'Prev_Close'): prev_close,
                ('Info', 'Div_Yield_Pct'): div_yield_pct,

                ('Open Metrics', 'Ex_Open'): ex_open,
                ('Open Metrics', 'Open_Price_Drop'): open_drop,
                ('Open Metrics', 'Open_Drop_Pct_of_Div'): open_drop_pct_div,

                ('Low Metrics', 'Ex_Low'): ex_low,
                ('Low Metrics', 'Low_Price_Drop'): low_drop,
                ('Low Metrics', 'Low_Drop_Pct_of_Div'): low_drop_pct_div,

                ('Market Adjusted', 'SPY_Adjusted_Drop'): adjusted_open_drop,
                ('Market Adjusted', 'Adj_Open_Drop_Pct_of_Div'): adj_open_drop_pct_div,
            })

    except Exception as e:
        print(f"Could not fetch data for {ticker}: {e}")

df_ex_dates = pd.DataFrame(ex_div_records)
df_ex_dates.columns = pd.MultiIndex.from_tuples(df_ex_dates.columns)

# Print processing summary and data verification
print(f"Scanned Tickers: {len(tickers)} / {len(sp500_df)}")
print(f"Total Ex-Dividend Events Captured: {len(df_ex_dates)}")
print(f"Unique Tickers with Dividend Events: {df_ex_dates[('Info', 'Ticker')].nunique() if not df_ex_dates.empty else 0}")

Scanned Tickers: 503 / 503
Total Ex-Dividend Events Captured: 3174
Unique Tickers with Dividend Events: 407


In [45]:
# 1. Select the percentage drop columns
open_pct = df_ex_dates[('Open Metrics', 'Open_Drop_Pct_of_Div')]
low_pct = df_ex_dates[('Low Metrics', 'Low_Drop_Pct_of_Div')]

# 2. Compute summary statistics
stats_data = {
    'Open Drop (% of Div)': {
        'Mean': open_pct.mean(),
        'Median': open_pct.median(),
        'Std Dev': open_pct.std(),
        'Count': open_pct.count()
    },
    'Low Drop (% of Div)': {
        'Mean': low_pct.mean(),
        'Median': low_pct.median(),
        'Std Dev': low_pct.std(),
        'Count': low_pct.count()
    }
}

# 3. Convert to DataFrame and format
stats_df = pd.DataFrame(stats_data)

# Round values for clean display
display(stats_df.round(2))

,Open Drop (% of Div),Low Drop (% of Div)
Mean,52.46,603.65
Median,89.68,271.50
Std Dev,1519.91,1899.27
Count,3174.00,3174.00
